# 0) Details About this notebook

 - This NoteBook is mainly running on Microsoft Fabric
 - This NoteBook is using the normal/default python environment provided on fabric not the PySpark environment
 - Before this step: I created a Lakehouse to Store the raw data and the cleaned data in -> Made 2 Schemas inside the Lakehouse's Tables section (ODS, STG) -> Uploaded the raw data manually to the Lakehouse's Files Section 
 - The target of this NoteBook is to move the data from the Lakehouse's File section to the Lakehouse's Table section espically the ODS schema that i created earlier
 - The raw data files will be stored in there main file format
 - We will make a small check if any of the files has an extra column due to the extra seperators added in the middle of the column values
 - We will make a small check if any of the files has an less column due to the missing separators in the middle of the column values
 - Add some Meta Data to each file

# 1) Downloading the Needed Packages

In [1]:
%pip install deltalake pyarrow openpyxl pandas datetime glob

StatementMeta(, 13718767-4c9d-4cc5-a0a5-c08c4fbedd54, 7, Finished, Available, Finished, True)

ERROR: Could not find a version that satisfies the requirement glob (from versions: none)
ERROR: No matching distribution found for glob

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



# 2) Knowing and Confirming the Paths to the Data

In [2]:
import pandas as pd
import os
import glob

# Get all files in the Files section
files_path = "/lakehouse/default/Files"
all_files = glob.glob(f"{files_path}/**/*", recursive=True)

# Filter out directories (keep only actual files)
files_list = []
for file_path in all_files:
    if os.path.isfile(file_path):
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path)
        file_ext = os.path.splitext(file_name)[1] or "No extension"
        files_list.append({
            "file_name": file_name,
            "full_path": file_path,
            "size_mb": round(file_size / (1024 * 1024), 2),
            "extension": file_ext
        })

df_files = pd.DataFrame(files_list)
print(f"Total files in Files section: {len(df_files)}")
display(df_files)

# Store paths in variables for later use
for idx, row in df_files.iterrows():
    print(f"{row['file_name'].replace('.', '_').replace('-', '_')}_path = '{row['full_path']}'")

StatementMeta(, 13718767-4c9d-4cc5-a0a5-c08c4fbedd54, 9, Finished, Available, Finished, False)

Total files in Files section: 4


SynapseWidget(Synapse.DataFrame, 3d73e7c7-4b5b-4835-ab53-b2766a840302)

employee_dirty_excel_xlsx_path = '/lakehouse/default/Files/employee_dirty_files_pre_ODS/employee_dirty_excel.xlsx'
employee_dirty_json_json_path = '/lakehouse/default/Files/employee_dirty_files_pre_ODS/employee_dirty_json.json'
employee_dirty_txt_txt_path = '/lakehouse/default/Files/employee_dirty_files_pre_ODS/employee_dirty_txt.txt'
employee_dirty_xml_xml_path = '/lakehouse/default/Files/employee_dirty_files_pre_ODS/employee_dirty_xml.xml'


# 3) Moving the Raw Excel Data From the Lakehouse to the ODS Layer

## Excel Extra Columns Detection

This function detects hidden extra columns in Excel files that `pd.read_excel()` silently ignores. Unlike CSV files that throw errors for mismatched columns, Excel files can contain data beyond the header column count that gets silently dropped during standard pandas reading.

### What It Does

- **Reads raw Excel data** using `openpyxl` to access the actual cell grid beyond the header-defined columns
- **Uses the header row as a template** to determine expected columns
- **Validates each row** by checking if data exists beyond the expected column count
- **Classifies each row** as GOOD (no extra columns) or BAD (extra columns detected)
- **Adds metadata** (`load_timestamp`, `source_location`, `file_type`) to all good records
- **Logs detailed error information** for rows with extra columns, including:
  - Row number where extra data was found
  - Expected vs actual column count
  - The extra values that would normally be silently dropped

### What It Does NOT Do

- ❌ Does NOT flag missing/null values as errors (these are handled in the cleaning stage)
- ❌ Does NOT validate data types or formats
- ❌ Does NOT modify the source file

### How Data is Classified

**GOOD Record:**
- No data beyond the expected column count
- Missing/null values within expected columns are acceptable

**BAD Record - EXTRA_COLUMNS:**
- Contains data in cells beyond the header column count
- These extra values are captured in the error log for investigation

### Example

| id | name | salary | (hidden) |
|----|------|--------|----------|
| 1  | John | 50000  | Extra    |

- Standard `pd.read_excel()` reads only: `id`, `name`, `salary`
- This function detects: `Extra` in the hidden 4th column
- Row is classified as BAD and logged to the error file

### Outputs

- **Good Records**: `EmployeeExcelData.xlsx` - Clean data with metadata columns
- **Error Records**: `EmployeeExcelData_ExtraColumns_Errors.xlsx` - Detailed log of rows with hidden extra data

### Why This Matters

Excel's silent dropping of extra column data can lead to data loss without any warning. This function ensures complete visibility into your data ingestion pipeline by capturing data that would otherwise be lost.


In [3]:
import pandas as pd
from openpyxl import load_workbook
from datetime import datetime
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType
from pyspark.sql.functions import col

def detect_extra_excel_columns(source, target_table, error_table, source_location=None, file_type="xlsx"):
    """
    Validates Excel rows against header template - ONLY for extra columns.
    Missing/null values are preserved as actual None/NaN.
    Saves data as delta tables in lakehouse.
    """
    if source_location is None:
        source_location = source
    
    load_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Step 1: Read Excel with openpyxl to get raw data
    wb = load_workbook(source, read_only=True)
    ws = wb.active
    
    # Extract header as template
    header = [cell.value if cell.value is not None else None for cell in ws[1]]
    # Remove trailing None values from header
    while header and header[-1] is None:
        header.pop()
    
    template_columns = header
    template_count = len(template_columns)
    
    print(f"📋 Template columns ({template_count}): {template_columns}")
    
    # Step 2: Process each row - ONLY check for extra columns
    good_records = []
    bad_records = []
    
    for row_idx, row in enumerate(ws.iter_rows(min_row=2, values_only=True), start=2):
        # Extract ALL values preserving None
        row_data = list(row)
        
        # Find the last non-None position (actual data extent)
        last_data_index = -1
        for i in range(len(row_data) - 1, -1, -1):
            if row_data[i] is not None:
                last_data_index = i
                break
        
        # If row is completely empty, skip it
        if last_data_index == -1:
            continue
        
        # Get actual row values (preserving None in between)
        actual_row_values = row_data[:last_data_index + 1]
        
        # Map data to template columns
        record = {}
        for i in range(min(template_count, len(actual_row_values))):
            # Convert all values to string to avoid type inference issues
            val = actual_row_values[i]
            # Keep None as None, convert everything else to string
            record[template_columns[i]] = None if val is None else str(val)
        
        # Fill remaining template columns with None
        for col in template_columns[len(actual_row_values):]:
            record[col] = None
        
        # Check for extra columns only (beyond template)
        has_extra_columns = last_data_index >= template_count
        
        if not has_extra_columns:
            # GOOD RECORD - no extra columns (missing values are preserved as None)
            record["load_timestamp"] = load_timestamp
            record["source_location"] = source_location
            record["file_type"] = file_type
            good_records.append(record)
        else:
            # BAD RECORD - has extra columns
            extra_values = actual_row_values[template_count:]
            extra_values = [v for v in extra_values if v is not None]
            
            error_info = {
                "row_number": str(row_idx),
                "expected_columns": str(template_count),
                "actual_columns": str(last_data_index + 1),
                "extra_values": ", ".join([str(v) for v in extra_values]) if extra_values else "None",
                "raw_data": str([v for v in actual_row_values]),
                "error_type": "EXTRA_COLUMNS",
                "load_timestamp": load_timestamp,
                "source_location": source_location
            }
            bad_records.append(error_info)
    
    wb.close()
    
    # Step 3: Create DataFrames
    df_good = pd.DataFrame(good_records) if good_records else pd.DataFrame(columns=template_columns + ["load_timestamp", "source_location", "file_type"])
    df_bad = pd.DataFrame(bad_records) if bad_records else pd.DataFrame()
    
    # Step 4: Save as Delta tables in lakehouse
    spark = SparkSession.getActiveSession()
    if spark is None:
        spark = SparkSession.builder.appName("ExcelDataValidation").getOrCreate()
    
    # Define schema to avoid type inference issues
    if not df_good.empty:
        # Build schema from DataFrame columns - all as StringType
        good_columns = df_good.columns.tolist()
        schema_good = StructType([
            StructField(col_name, StringType(), True) for col_name in good_columns
        ])
        
        # Convert to Spark DataFrame with explicit schema
        spark_good = spark.createDataFrame(
            spark.sparkContext.parallelize(df_good.values.tolist()),
            schema_good
        )
        
        # Save good records as table
        spark_good.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"✅ Good records saved to table: {target_table}")
        print(f"   Row count: {spark_good.count()}")
    else:
        print(f"⚠️ No good records to save")
    
    if not df_bad.empty:
        # Build schema for bad records - all as StringType
        bad_columns = df_bad.columns.tolist()
        schema_bad = StructType([
            StructField(col_name, StringType(), True) for col_name in bad_columns
        ])
        
        # Convert to Spark DataFrame with explicit schema
        spark_bad = spark.createDataFrame(
            spark.sparkContext.parallelize(df_bad.values.tolist()),
            schema_bad
        )
        
        # Save bad records as table
        spark_bad.write.format("delta").mode("overwrite").saveAsTable(error_table)
        print(f"✅ Bad records saved to table: {error_table}")
        print(f"   Row count: {spark_bad.count()}")
    else:
        print(f"⚠️ No bad records to save")
    
    # Step 5: Summary
    total_processed = len(good_records) + len(bad_records)
    print(f"\n{'='*50}")
    print(f"✅ EXTRA COLUMNS DETECTION COMPLETE")
    print(f"   Total rows processed: {total_processed}")
    print(f"   Good records (no extra columns): {len(good_records)} → {target_table}")
    print(f"   Bad records (extra columns found): {len(bad_records)} → {error_table}")
    
    if not df_bad.empty:
        print(f"\n📊 Rows with hidden extra data:")
        for _, row in df_bad.iterrows():
            print(f"   Row {row['row_number']}: {row['extra_values']}")
    
    return df_good, df_bad

# Usage - Table names in lakehouse
source = "/lakehouse/default/Files/employee_dirty_files_pre_ODS/employee_dirty_excel.xlsx"
target_table = "ODS.Employee_Excel_Data"  # Table name in lakehouse
error_table = "ODS.Employee_Excel_Data_Extra_Columns_Errors"  # Table name in lakehouse

df_good, df_bad = detect_extra_excel_columns(source, target_table, error_table)

StatementMeta(, 13718767-4c9d-4cc5-a0a5-c08c4fbedd54, 10, Finished, Available, Finished, False)

📋 Template columns (12): ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level']
✅ Good records saved to table: ODS.Employee_Excel_Data
   Row count: 1000
⚠️ No bad records to save

✅ EXTRA COLUMNS DETECTION COMPLETE
   Total rows processed: 1000
   Good records (no extra columns): 1000 → ODS.Employee_Excel_Data
   Bad records (extra columns found): 0 → ODS.Employee_Excel_Data_Extra_Columns_Errors


In [16]:
# Get or create Spark session
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("ReadODSTables").getOrCreate()

# Display good records from ODS table
print("="*50)
print("GOOD RECORDS (EmployeeExcelData)")
print("="*50)

# Read from the Delta table
df_good_spark = spark.table("ODS.Employee_Excel_Data")

# Convert to pandas for display (limit rows for performance)
df_good_pandas = df_good_spark.limit(1000).toPandas()

print(f"Total rows: {df_good_spark.count()}")
print(f"Columns: {df_good_pandas.columns.tolist()}")
print("\nFirst 5 rows:")
display(df_good_pandas.head())

StatementMeta(, 5550a4ea-07b0-45ee-ba32-9663c47d2ba5, 39, Finished, Available, Finished, False)

GOOD RECORDS (EmployeeExcelData)
Total rows: 1000
Columns: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level', 'load_timestamp', 'source_location', 'file_type']

First 5 rows:


SynapseWidget(Synapse.DataFrame, 164f1684-1682-480b-ab89-a96346944157)

In [17]:
print("\n" + "="*50)
print("BAD RECORDS (EmployeeExcelData_ExtraColumns_Errors)")
print("="*50)

# Check if the error table exists and has data
try:
    # Read from the Delta table
    df_bad_spark = spark.table("ODS.Employee_Excel_Data_Extra_Columns_Errors")
    
    # Check if table has data
    if df_bad_spark.count() > 0:
        df_bad_pandas = df_bad_spark.limit(1000).toPandas()
        print(f"Total rows: {df_bad_spark.count()}")
        print(f"Columns: {df_bad_pandas.columns.tolist()}")
        print("\nFirst 5 rows:")
        display(df_bad_pandas.head())
    else:
        print("✅ No error records - all data was clean (no extra columns found)")
        
except Exception as e:
    # Table doesn't exist or other error
    print("✅ No error records - all data was clean (no extra columns found)")
    # print(f"(Debug: {str(e)})")

StatementMeta(, 5550a4ea-07b0-45ee-ba32-9663c47d2ba5, 44, Finished, Available, Finished, False)


BAD RECORDS (EmployeeExcelData_ExtraColumns_Errors)
✅ No error records - all data was clean (no extra columns found)


# 4) Moving the Raw TXT Data From the Lakehouse to the ODS Layer

## Data Processing Pipeline: Employee Text File Validation

### Overview
This notebook implements a robust ETL (Extract, Transform, Load) process to read a pipe-delimited employee data file, validate its structure, and separate clean records from erroneous ones.

### Why This Approach?
The source data file (`employee_dirty_txt.txt`) contains inconsistencies that need to be handled before loading into the data lakehouse:
- **Missing columns** in some rows
- **Extra columns** in other rows (e.g., line 126 has 13 fields instead of 12)
- **Embedded commas** in fields (e.g., names like "Martinez, Hamad")
- **Inconsistent formatting** across records

### The Algorithm: 4-Phase Validation

#### Phase 1: File Inspection
- Opens the file and reads it line by line
- Skips empty lines to avoid false errors
- First non-empty line is identified as the **header row**
- Counts header fields to determine the **expected column count** (12 fields)
- Prepares metadata columns: `load_timestamp`, `source_location`, `file_type`

#### Phase 2: Row-by-Row Validation
For each subsequent row (data row):
- Splits the line by the pipe separator (`|`)
- Counts the number of fields
- Compares against the expected column count from Phase 1

#### Phase 3: Classification
Each row is classified into one of three categories:
- **GOOD ROW**: Field count matches expected (12 fields)
- **MISSING_COLUMNS**: Field count is less than expected (incomplete data)
- **EXTRA_COLUMNS**: Field count exceeds expected (additional pipe delimiters in data)

#### Phase 4: Error Logging
For each problematic row, the following is captured:
- **Line number** (for debugging)
- **Expected fields** (12)
- **Actual fields** (actual count)
- **Original line content** (preserved for inspection)
- **Error type** (MISSING_COLUMNS or EXTRA_COLUMNS)
- **Metadata**: load timestamp, source location, and file type

### Data Transformation

#### Good Records
- Original 12 columns are preserved
- Three metadata columns are appended:
  - `load_timestamp`: When the data was processed
  - `source_location`: Original file path
  - `file_type`: File format (txt)
- Saved to: `/Tables/ODS/EmployeeTextData.txt`

#### Bad Records
- Includes all the error details captured in Phase 4
- Allows for downstream investigation and data quality tracking
- Saved to: `/Tables/ODS/EmployeeTextData_Errors.txt`

### Output Summary
The process outputs:
1. **EmployeeTextData.txt**: Clean records ready for consumption
2. **EmployeeTextData_Errors.txt**: Error log with detailed issue tracking
3. **Console summary**: Statistics showing total records, good records, and error breakdown

### Benefits
- **Data Quality**: Identifies and isolates problematic records
- **Audit Trail**: Complete error logging for troubleshooting
- **Flexibility**: Works with any delimiter (pipe, comma, tab)
- **Metadata Tracking**: Source and timestamp information for lineage
- **Non-destructive**: Original file remains unchanged
- **Scalable**: Processes line-by-line, handling large files efficiently


In [1]:
import pandas as pd
from datetime import datetime
import os
import re
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col

def sanitize_column_name(name):
    """
    Sanitize column names to be Delta-compliant.
    Replaces spaces and special characters with underscores.
    """
    # Replace spaces and special characters with underscore
    name = re.sub(r'[ ,;{}()\n\t=]', '_', name)
    # Remove multiple underscores
    name = re.sub(r'_+', '_', name)
    # Remove leading/trailing underscores
    name = name.strip('_')
    # If name starts with number, prefix with underscore
    if name and name[0].isdigit():
        name = '_' + name
    return name

def read_delimited_file_to_tables(file_path, target_table, error_table, separator='|', source_location=None, file_type=None):
    """
    Robust delimited file reader that detects column count issues.
    Saves data as Delta tables in lakehouse.
    
    Parameters:
    - file_path: Path to the input file
    - target_table: Target Delta table name (e.g., "ODS.EmployeeTextData")
    - error_table: Error Delta table name (e.g., "ODS.EmployeeTextData_Errors")
    - separator: Delimiter character (default: '|')
    - source_location: Source location metadata (default: file path)
    - file_type: File type metadata (default: 'txt')
    
    Returns:
    - good_rows: List of clean rows (each row as list of fields)
    - bad_rows: List of error records with metadata
    - header: List of column names
    """
    
    # Set default metadata
    if source_location is None:
        source_location = file_path
    if file_type is None:
        file_type = 'txt'
    
    load_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    good_rows = []
    bad_rows = []
    header = None
    expected_column_count = None
    original_header = None
    
    # Phase 1: File Inspection
    with open(file_path, 'r') as f:
        line_num = 0
        for line in f:
            line = line.strip()
            line_num += 1
            
            # Skip empty lines
            if not line:
                continue
            
            # Split by separator
            fields = line.split(separator)
            
            # First non-empty line is header
            if header is None:
                original_header = fields
                # Sanitize header columns for Delta compatibility
                header = [sanitize_column_name(col) for col in fields]
                expected_column_count = len(header)
                # Add metadata columns to header
                header_with_meta = header + ['load_timestamp', 'source_location', 'file_type']
                continue
            
            # Phase 2: Row-by-Row Validation
            actual_column_count = len(fields)
            
            # Phase 3: Classification
            if actual_column_count == expected_column_count:
                # GOOD ROW - add metadata
                row_with_meta = fields + [load_timestamp, source_location, file_type]
                good_rows.append(row_with_meta)
            else:
                # Phase 4: Error Logging
                if actual_column_count < expected_column_count:
                    error_type = "MISSING_COLUMNS"
                else:
                    error_type = "EXTRA_COLUMNS"
                
                bad_rows.append({
                    'line_number': line_num,
                    'expected_fields': expected_column_count,
                    'actual_fields': actual_column_count,
                    'original_line': line,
                    'error_type': error_type,
                    'load_timestamp': load_timestamp,
                    'source_location': source_location,
                    'file_type': file_type
                })
    
    # Get or create Spark session
    spark = SparkSession.getActiveSession()
    if spark is None:
        spark = SparkSession.builder.appName("DelimitedFileValidation").getOrCreate()
    
    # Save good records as Delta table
    if good_rows:
        # Create pandas DataFrame for good rows
        df_good_pd = pd.DataFrame(good_rows, columns=header_with_meta)
        
        # Convert to Spark DataFrame with explicit schema (all as StringType)
        good_columns = df_good_pd.columns.tolist()
        schema_good = StructType([
            StructField(col_name, StringType(), True) for col_name in good_columns
        ])
        
        spark_good = spark.createDataFrame(
            spark.sparkContext.parallelize(df_good_pd.values.tolist()),
            schema_good
        )
        
        # Save as Delta table
        spark_good.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"✅ Good records saved to table: {target_table}")
        print(f"   Row count: {spark_good.count()}")
        print(f"   Columns: {good_columns}")
    else:
        print("⚠️ No good records to save")
    
    # Save bad records as Delta table
    if bad_rows:
        # Create pandas DataFrame for bad rows
        df_bad_pd = pd.DataFrame(bad_rows)
        
        # Convert to Spark DataFrame with explicit schema (all as StringType)
        bad_columns = df_bad_pd.columns.tolist()
        schema_bad = StructType([
            StructField(col_name, StringType(), True) for col_name in bad_columns
        ])
        
        spark_bad = spark.createDataFrame(
            spark.sparkContext.parallelize(df_bad_pd.values.tolist()),
            schema_bad
        )
        
        # Save as Delta table
        spark_bad.write.format("delta").mode("overwrite").saveAsTable(error_table)
        print(f"✅ Error records saved to table: {error_table}")
        print(f"   Row count: {spark_bad.count()}")
    else:
        print("✅ No bad records to save")
    
    # Summary
    print("\n" + "="*50)
    print("SUMMARY:")
    print("="*50)
    print(f"Total records processed: {len(good_rows) + len(bad_rows)}")
    print(f"Good records: {len(good_rows)} → {target_table}")
    print(f"Error records: {len(bad_rows)} → {error_table}")
    
    if bad_rows:
        # Get error breakdown from bad_rows
        error_types = {}
        for row in bad_rows:
            error_types[row['error_type']] = error_types.get(row['error_type'], 0) + 1
        print("\nError breakdown:")
        for error_type, count in error_types.items():
            print(f"  {error_type}: {count}")
    
    return good_rows, bad_rows, header_with_meta


# Define paths and table names
source = "/lakehouse/default/Files/employee_dirty_files_pre_ODS/employee_dirty_txt.txt"
target_table = "ODS.Employee_Text_Data"  # Delta table name for good records
error_table = "ODS.Employee_Text_Data_Errors"  # Delta table name for error records

# Process the file and save to Delta tables
print("Processing file...")
good_rows, bad_rows, header = read_delimited_file_to_tables(
    file_path=source,
    target_table=target_table,
    error_table=error_table,
    separator='|',
    source_location="Files/employee_dirty_files_pre_ODS/employee_dirty_txt.txt",
    file_type="txt"
)

StatementMeta(, 707a8dac-408d-4be1-ba9f-2180604d21a8, 3, Finished, Available, Finished, False)

Processing file...
✅ Good records saved to table: ODS.Employee_Text_Data
   Row count: 993
   Columns: ['emp_id', 'full_name', 'fname', 'lname', 'birth_date', 'dept_name', 'country', 'monthly_salary', 'sex', 'job_title', 'employee_status', 'education', 'load_timestamp', 'source_location', 'file_type']
✅ Error records saved to table: ODS.Employee_Text_Data_Errors
   Row count: 7

SUMMARY:
Total records processed: 1000
Good records: 993 → ODS.Employee_Text_Data
Error records: 7 → ODS.Employee_Text_Data_Errors

Error breakdown:
  EXTRA_COLUMNS: 5
  MISSING_COLUMNS: 2


In [2]:
# Get or create Spark session
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("ReadODSTables").getOrCreate()

print("="*50)
print("GOOD RECORDS (EmployeeTextData)")
print("="*50)

# Read from the Delta table
df_good_spark = spark.table("ODS.Employee_Text_Data")

# Show basic info
print(f"Total rows: {df_good_spark.count()}")
print(f"Columns: {df_good_spark.columns}")

# Show sample data
print("\nFirst 5 rows:")
df_good_pandas = df_good_spark.limit(5).toPandas()
display(df_good_pandas)

StatementMeta(, 707a8dac-408d-4be1-ba9f-2180604d21a8, 4, Finished, Available, Finished, False)

GOOD RECORDS (EmployeeTextData)
Total rows: 993
Columns: ['emp_id', 'full_name', 'fname', 'lname', 'birth_date', 'dept_name', 'country', 'monthly_salary', 'sex', 'job_title', 'employee_status', 'education', 'load_timestamp', 'source_location', 'file_type']

First 5 rows:


SynapseWidget(Synapse.DataFrame, 420b0ebe-040e-4170-8e47-9aca3a213bf0)

In [3]:
print("\n" + "="*50)
print("BAD RECORDS (EmployeeTextData_Errors)")
print("="*50)

try:
    # Read from the Delta table
    df_bad_spark = spark.table("ODS.Employee_Text_Data_Errors")
    
    # Check if table has data
    if df_bad_spark.count() > 0:
        df_bad_pandas = df_bad_spark.limit(1000).toPandas()
        print(f"Total rows: {df_bad_spark.count()}")
        print(f"Columns: {df_bad_pandas.columns.tolist()}")
        print("\nFirst 5 rows:")
        display(df_bad_pandas.head())
        
        # Show error breakdown
        print("\n📊 Error breakdown:")
        df_bad_spark.groupBy("error_type").count().show()
    else:
        print("✅ No error records - all data was clean")
        
except Exception as e:
    print("✅ No error records - all data was clean")
    # print(f"(Debug: {str(e)})")

StatementMeta(, 707a8dac-408d-4be1-ba9f-2180604d21a8, 5, Finished, Available, Finished, False)


BAD RECORDS (EmployeeTextData_Errors)
Total rows: 7
Columns: ['line_number', 'expected_fields', 'actual_fields', 'original_line', 'error_type', 'load_timestamp', 'source_location', 'file_type']

First 5 rows:


SynapseWidget(Synapse.DataFrame, e845f171-a6de-43ac-a725-9774cc0f3b88)


📊 Error breakdown:
+---------------+-----+
|     error_type|count|
+---------------+-----+
|  EXTRA_COLUMNS|    5|
|MISSING_COLUMNS|    2|
+---------------+-----+



# 5) Moving the Raw XML Data From the Lakehouse to the ODS Layer

## XML Data Validation and Processing

This function reads an XML file containing employee data, validates each employee record against a template structure, and separates valid records from invalid ones.

### What It Does

- **Parses the XML file** and extracts all `<Employee>` elements
- **Uses the first employee** as a template to determine expected tags (e.g., id, full_name, f_name, l_name, dob, dept, country, salary, gender, job_title, employment_status, education_level)
- **Validates each employee** by checking:
  - All expected tags are present (no missing tags)
  - No unexpected tags exist (no extra tags)
- **Classifies** each employee as GOOD or BAD
- **Adds metadata** (load_timestamp, source_location, file_type) to all good records
- **Logs detailed error information** for bad records, including:
  - Which tags are missing
  - Which tags are extra
  - The raw XML content of the problematic record

### How Data is Classified

**GOOD Record:**
- Must contain all expected tags from the template
- Must have exactly the same number of tags as the template
- Must not contain any unexpected tags

**BAD Record - MISSING_TAGS:**
- One or more expected tags are missing
- Example: Employee is missing the `<employment_status>` tag

**BAD Record - EXTRA_TAGS:**
- Contains tags that are not in the template
- Example: Employee has an additional `<middle_name>` tag

**BAD Record - MISSING_TAGS | EXTRA_TAGS:**
- Both issues present simultaneously
- Missing some required tags AND has unexpected tags

### Outputs

- **Good Records**: `EmployeeXMLData.xlsx` - Clean employee data with metadata columns
- **Error Records**: `EmployeeXMLData_Errors.xlsx` - Detailed error log for investigation

### Summary

The function processes all employees, displays validation statistics, and saves both clean and erroneous records for further analysis or correction.


In [25]:
import pandas as pd
from datetime import datetime
import os
import re
import xml.etree.ElementTree as ET
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

def sanitize_column_name(name):
    """
    Sanitize column names to be Delta-compliant.
    Replaces spaces and special characters with underscores.
    """
    # Replace spaces and special characters with underscore
    name = re.sub(r'[ ,;{}()\n\t=]', '_', name)
    # Remove multiple underscores
    name = re.sub(r'_+', '_', name)
    # Remove leading/trailing underscores
    name = name.strip('_')
    # If name starts with number, prefix with underscore
    if name and name[0].isdigit():
        name = '_' + name
    return name

def read_xml_with_validation(file_path, target_table, error_table, source_location=None, file_type="xml"):
    """
    Robust XML reader that detects missing or extra tags.
    Saves data as Delta tables in lakehouse.
    """
    
    # Set default metadata
    if source_location is None:
        source_location = file_path
    
    load_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    good_rows = []
    bad_rows = []
    expected_tags = None
    sanitized_expected_tags = None
    
    # Phase 1: File Inspection
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
    except Exception as e:
        print(f"Error parsing XML: {e}")
        return [], [], []
    
    # Find all Employee elements
    employees = root.findall('Employee')
    
    if not employees:
        print("No Employee elements found")
        return [], [], []
    
    # Use first Employee as template for expected tags
    expected_tags = [child.tag for child in employees[0]]
    # Sanitize tags for Delta compatibility
    sanitized_expected_tags = [sanitize_column_name(tag) for tag in expected_tags]
    expected_count = len(expected_tags)
    expected_tags_set = set(expected_tags)
    
    print(f"Expected tags ({expected_count}): {expected_tags}")
    print(f"Sanitized tags: {sanitized_expected_tags}")
    print(f"Total employees: {len(employees)}")
    print("-" * 50)
    
    # Phase 2 & 3: Row-by-Row Validation & Classification
    for idx, employee in enumerate(employees, 1):
        # Get all direct child tags
        child_tags = [child.tag for child in employee]
        child_tags_set = set(child_tags)
        actual_count = len(child_tags)
        
        # Check for missing tags
        missing_tags = list(expected_tags_set - child_tags_set)
        
        # Check for extra tags
        extra_tags = list(child_tags_set - expected_tags_set)
        
        # Phase 3: Classification
        if actual_count == expected_count and not extra_tags:
            # GOOD ROW - convert to dict with metadata using sanitized column names
            row_dict = {}
            for child in employee:
                sanitized_tag = sanitize_column_name(child.tag)
                row_dict[sanitized_tag] = child.text
            row_dict['load_timestamp'] = load_timestamp
            row_dict['source_location'] = source_location
            row_dict['file_type'] = file_type
            good_rows.append(row_dict)
        else:
            # Phase 4: Error Logging
            error_type = []
            if missing_tags:
                error_type.append("MISSING_TAGS")
            if extra_tags:
                error_type.append("EXTRA_TAGS")
            
            # Convert employee to string for logging
            employee_str = ET.tostring(employee, encoding='unicode')
            
            bad_rows.append({
                'employee_number': str(idx),
                'expected_tags': str(expected_tags),
                'expected_count': str(expected_count),
                'actual_tags': str(child_tags),
                'actual_count': str(actual_count),
                'missing_tags': str(missing_tags),
                'extra_tags': str(extra_tags),
                'error_type': " | ".join(error_type),
                'raw_xml': employee_str[:500] + "..." if len(employee_str) > 500 else employee_str,
                'load_timestamp': load_timestamp,
                'source_location': source_location,
                'file_type': file_type
            })
    
    # Get or create Spark session
    spark = SparkSession.getActiveSession()
    if spark is None:
        spark = SparkSession.builder.appName("XMLValidation").getOrCreate()
    
    # Save good records as Delta table
    if good_rows:
        # Create pandas DataFrame for good rows
        df_good_pd = pd.DataFrame(good_rows)
        
        # Convert to Spark DataFrame with explicit schema (all as StringType)
        good_columns = df_good_pd.columns.tolist()
        schema_good = StructType([
            StructField(col_name, StringType(), True) for col_name in good_columns
        ])
        
        spark_good = spark.createDataFrame(
            spark.sparkContext.parallelize(df_good_pd.values.tolist()),
            schema_good
        )
        
        # Save as Delta table
        spark_good.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"✅ Good records saved to table: {target_table}")
        print(f"   Row count: {spark_good.count()}")
        print(f"   Columns: {good_columns}")
    else:
        print("⚠️ No good records to save")
    
    # Save bad records as Delta table
    if bad_rows:
        # Create pandas DataFrame for bad rows
        df_bad_pd = pd.DataFrame(bad_rows)
        
        # Convert to Spark DataFrame with explicit schema (all as StringType)
        bad_columns = df_bad_pd.columns.tolist()
        schema_bad = StructType([
            StructField(col_name, StringType(), True) for col_name in bad_columns
        ])
        
        spark_bad = spark.createDataFrame(
            spark.sparkContext.parallelize(df_bad_pd.values.tolist()),
            schema_bad
        )
        
        # Save as Delta table
        spark_bad.write.format("delta").mode("overwrite").saveAsTable(error_table)
        print(f"✅ Error records saved to table: {error_table}")
        print(f"   Row count: {spark_bad.count()}")
    else:
        print("✅ No bad records to save")
    
    # Summary
    print("\n" + "="*50)
    print("SUMMARY:")
    print("="*50)
    print(f"Total employees processed: {len(good_rows) + len(bad_rows)}")
    print(f"Good records: {len(good_rows)} → {target_table}")
    print(f"Error records: {len(bad_rows)} → {error_table}")
    
    if bad_rows:
        # Get error breakdown from bad_rows
        error_types = {}
        for row in bad_rows:
            error_types[row['error_type']] = error_types.get(row['error_type'], 0) + 1
        print("\nError breakdown:")
        for error_type, count in error_types.items():
            print(f"  {error_type}: {count}")
    
    return good_rows, bad_rows, expected_tags


# ============== MAIN LOGIC ==============

source = "/lakehouse/default/Files/employee_dirty_files_pre_ODS/employee_dirty_xml.xml"
target_table = "ODS.Employee_XML_Data"  # Delta table name for good records
error_table = "ODS.Employee_XML_Data_Errors"  # Delta table name for error records

# Process XML file and save to Delta tables
print("Processing XML file...")
good_rows, bad_rows, expected_tags = read_xml_with_validation(
    file_path=source,
    target_table=target_table,
    error_table=error_table,
    source_location="Files/employee_dirty_files_pre_ODS/employee_dirty_xml.xml",
    file_type="xml"
)

StatementMeta(, 5550a4ea-07b0-45ee-ba32-9663c47d2ba5, 57, Finished, Available, Finished, False)

Processing XML file...
Expected tags (12): ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level']
Sanitized tags: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level']
Total employees: 1000
--------------------------------------------------
✅ Good records saved to table: ODS.Employee_XML_Data
   Row count: 800
   Columns: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level', 'load_timestamp', 'source_location', 'file_type']
✅ Error records saved to table: ODS.Employee_XML_Data_Errors
   Row count: 200

SUMMARY:
Total employees processed: 1000
Good records: 800 → ODS.Employee_XML_Data
Error records: 200 → ODS.Employee_XML_Data_Errors

Error breakdown:
  MISSING_TAGS: 117
  MISSING_TAGS | EXTRA_TAGS: 83


In [28]:
# Get or create Spark session
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("ReadODSTables").getOrCreate()

print("="*50)
print("GOOD RECORDS (Employee_XML_Data)")
print("="*50)

# Read from the Delta table
df_good_spark = spark.table("ODS.Employee_XML_Data")

# Show basic info
total_good_rows = df_good_spark.count()
print(f"Total rows: {total_good_rows}")
print(f"Columns: {df_good_spark.columns}")

# Show sample data
print("\nFirst 5 rows:")
df_good_pandas = df_good_spark.limit(5).toPandas()
display(df_good_pandas)


StatementMeta(, 5550a4ea-07b0-45ee-ba32-9663c47d2ba5, 60, Finished, Available, Finished, False)

GOOD RECORDS (Employee_XML_Data)
Total rows: 800
Columns: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level', 'load_timestamp', 'source_location', 'file_type']

First 5 rows:


SynapseWidget(Synapse.DataFrame, 97382594-4619-4f6f-9b22-63159e1a4c16)

In [26]:
print("\n" + "="*50)
print("BAD RECORDS (EmployeeXMLData_Errors)")
print("="*50)

try:
    # Read from the Delta table
    df_bad_spark = spark.table("ODS.Employee_XML_Data_Errors")
    
    # Check if table has data
    if df_bad_spark.count() > 0:
        total_bad_rows = df_bad_spark.count()
        print(f"Total rows: {total_bad_rows}")
        print(f"Columns: {df_bad_spark.columns}")
        
        print("\nFirst 5 rows:")
        df_bad_pandas = df_bad_spark.limit(5).toPandas()
        display(df_bad_pandas)
        
        print("\n📊 Error breakdown:")
        display(df_bad_spark.groupBy("error_type").count().toPandas())
        
        print("\n📊 Missing tags breakdown:")
        display(df_bad_spark.filter(col("missing_tags") != "[]").groupBy("missing_tags").count().toPandas())
        
        print("\n📊 Extra tags breakdown:")
        display(df_bad_spark.filter(col("extra_tags") != "[]").groupBy("extra_tags").count().toPandas())

    else:
        print("✅ No error records - all data was clean")
        
except Exception as e:
    print("✅ No error records - all data was clean")

StatementMeta(, 5550a4ea-07b0-45ee-ba32-9663c47d2ba5, 58, Finished, Available, Finished, False)


BAD RECORDS (EmployeeXMLData_Errors)
Total rows: 200
Columns: ['employee_number', 'expected_tags', 'expected_count', 'actual_tags', 'actual_count', 'missing_tags', 'extra_tags', 'error_type', 'raw_xml', 'load_timestamp', 'source_location', 'file_type']

First 5 rows:


SynapseWidget(Synapse.DataFrame, 9c4846a6-b173-4538-9f3e-2cfadb9ebe52)


📊 Error breakdown:


SynapseWidget(Synapse.DataFrame, 3e508f91-7874-4000-a9ca-4d01d69ffd29)


📊 Missing tags breakdown:


SynapseWidget(Synapse.DataFrame, 5c3ec293-453e-4532-ada3-ebfd63f62daf)


📊 Extra tags breakdown:


SynapseWidget(Synapse.DataFrame, dacb9817-da98-432a-83c9-bee864792a97)

# 6) Moving the Raw JSON Data From the Lakehouse to the ODS Layer

## JSON Data Validation and Processing

This function reads a JSON file containing employee data, validates each employee record against a template structure, and separates valid records from invalid ones.

### What It Does

- **Parses the JSON file** and extracts all employee records from the `employees` array
- **Uses the first employee** as a template to determine expected fields (keys)
- **Validates each employee** by checking:
  - All expected fields are present (no missing fields)
  - No unexpected fields exist (no extra fields)
- **Classifies** each employee as GOOD or BAD
- **Adds metadata** (load_timestamp, source_location, file_type) to all good records
- **Logs detailed error information** for bad records, including:
  - Which fields are missing
  - Which fields are extra
  - The raw record data for investigation

### How Data is Classified

**GOOD Record:**
- Must contain all expected fields from the template
- Must have exactly the same number of fields as the template
- Must not contain any unexpected fields

**BAD Record - MISSING_FIELDS:**
- One or more expected fields are missing
- Example: Employee record is missing the `employment_status` field

**BAD Record - EXTRA_FIELDS:**
- Contains fields that are not in the template
- Example: Employee record has an additional `middle_name` field

**BAD Record - MISSING_FIELDS | EXTRA_FIELDS:**
- Both issues present simultaneously
- Missing some required fields AND has unexpected fields

### Outputs

- **Good Records**: `EmployeeJSONData.json` - Clean employee data with metadata fields
- **Error Records**: `EmployeeJSONData_Errors.json` - Detailed error log for investigation

### Summary

The function processes all employees, displays validation statistics, and saves both clean and erroneous records for further analysis or correction.


In [30]:
import pandas as pd
import json
from datetime import datetime
import os
import re
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType

def sanitize_column_name(name):
    """
    Sanitize column names to be Delta-compliant.
    """
    name = re.sub(r'[ ,;{}()\n\t=]', '_', name)
    name = re.sub(r'_+', '_', name)
    name = name.strip('_')
    if name and name[0].isdigit():
        name = '_' + name
    return name

def read_json_with_validation(file_path, target_table, error_table, source_location=None, file_type="json"):
    """
    Robust JSON reader that detects missing or extra fields.
    Saves data as Delta tables in lakehouse.
    """
    
    if source_location is None:
        source_location = file_path
    
    load_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    good_records = []
    bad_records = []
    
    # Phase 1: File Inspection
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        return [], []
    
    employees = data.get("employees", [])
    
    if not employees:
        print("No employee records found")
        return [], []
    
    # Use first record as template
    expected_keys = set(employees[0].keys())
    expected_count = len(expected_keys)
    
    # Create sanitized expected keys for delta table
    sanitized_keys = {sanitize_column_name(k) for k in expected_keys}
    
    print(f"Expected fields ({expected_count}): {sorted(expected_keys)}")
    print(f"Total employees: {len(employees)}")
    print("-" * 50)
    
    # Phase 2 & 3: Validation & Classification
    for idx, record in enumerate(employees, 1):
        record_keys = set(record.keys())
        actual_count = len(record_keys)
        
        # Check for missing fields
        missing_fields = expected_keys - record_keys
        
        # Check for extra fields
        extra_fields = record_keys - expected_keys
        
        # Phase 3: Classification
        if actual_count == expected_count and len(extra_fields) == 0:
            # GOOD ROW - add metadata using sanitized keys
            sanitized_record = {}
            for k, v in record.items():
                sanitized_record[sanitize_column_name(k)] = v
            sanitized_record['load_timestamp'] = load_timestamp
            sanitized_record['source_location'] = source_location
            sanitized_record['file_type'] = file_type
            good_records.append(sanitized_record)
        else:
            # Phase 4: Error Logging
            error_type = []
            if missing_fields:
                error_type.append("MISSING_FIELDS")
            if extra_fields:
                error_type.append("EXTRA_FIELDS")
            
            bad_records.append({
                'record_number': str(idx),
                'expected_count': str(expected_count),
                'actual_count': str(actual_count),
                'missing_fields': str(list(missing_fields)) if missing_fields else "None",
                'extra_fields': str(list(extra_fields)) if extra_fields else "None",
                'error_type': " | ".join(error_type),
                'record_data': str(record)[:500] + "..." if len(str(record)) > 500 else str(record),
                'load_timestamp': load_timestamp,
                'source_location': source_location,
                'file_type': file_type
            })
    
    # Get or create Spark session
    spark = SparkSession.getActiveSession()
    if spark is None:
        spark = SparkSession.builder.appName("JSONValidation").getOrCreate()
    
    # Save good records as Delta table
    if good_records:
        df_good_pd = pd.DataFrame(good_records)
        good_columns = df_good_pd.columns.tolist()
        schema_good = StructType([
            StructField(col_name, StringType(), True) for col_name in good_columns
        ])
        spark_good = spark.createDataFrame(
            spark.sparkContext.parallelize(df_good_pd.values.tolist()),
            schema_good
        )
        spark_good.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"✅ Good records saved to table: {target_table}")
        print(f"   Row count: {spark_good.count()}")
    else:
        print("⚠️ No good records to save")
    
    # Save bad records as Delta table
    if bad_records:
        df_bad_pd = pd.DataFrame(bad_records)
        bad_columns = df_bad_pd.columns.tolist()
        schema_bad = StructType([
            StructField(col_name, StringType(), True) for col_name in bad_columns
        ])
        spark_bad = spark.createDataFrame(
            spark.sparkContext.parallelize(df_bad_pd.values.tolist()),
            schema_bad
        )
        spark_bad.write.format("delta").mode("overwrite").saveAsTable(error_table)
        print(f"✅ Error records saved to table: {error_table}")
        print(f"   Row count: {spark_bad.count()}")
    else:
        print("✅ No bad records to save")
    
    # Summary
    print("\n" + "="*50)
    print("SUMMARY:")
    print("="*50)
    print(f"Total records processed: {len(good_records) + len(bad_records)}")
    print(f"Good records: {len(good_records)} → {target_table}")
    print(f"Error records: {len(bad_records)} → {error_table}")
    
    if bad_records:
        error_types = {}
        for row in bad_records:
            error_types[row['error_type']] = error_types.get(row['error_type'], 0) + 1
        print("\nError breakdown:")
        for error_type, count in error_types.items():
            print(f"  {error_type}: {count}")
    
    return good_records, bad_records


# ============== MAIN LOGIC ==============

source = "/lakehouse/default/Files/employee_dirty_files_pre_ODS/employee_dirty_json.json"
target_table = "ODS.Employee_JSON_Data"
error_table = "ODS.Employee_JSON_Data_Errors"

print("Processing JSON file...")
good_records, bad_records = read_json_with_validation(
    file_path=source,
    target_table=target_table,
    error_table=error_table,
    source_location="Files/employee_dirty_files_pre_ODS/employee_dirty_json.json",
    file_type="json"
)

StatementMeta(, 5550a4ea-07b0-45ee-ba32-9663c47d2ba5, 62, Finished, Available, Finished, False)

Processing JSON file...
Expected fields (12): ['country', 'dept', 'dob', 'education_level', 'employment_status', 'f_name', 'full_name', 'gender', 'id', 'job_title', 'l_name', 'salary']
Total employees: 1000
--------------------------------------------------
✅ Good records saved to table: ODS.Employee_JSON_Data
   Row count: 986
✅ Error records saved to table: ODS.Employee_JSON_Data_Errors
   Row count: 14

SUMMARY:
Total records processed: 1000
Good records: 986 → ODS.Employee_JSON_Data
Error records: 14 → ODS.Employee_JSON_Data_Errors

Error breakdown:
  MISSING_FIELDS | EXTRA_FIELDS: 14


In [32]:
print("="*50)
print("GOOD RECORDS (EmployeeJSONData)")
print("="*50)

df_good_spark = spark.table("ODS.Employee_JSON_Data")
print(f"Total rows: {df_good_spark.count()}")
print(f"Columns: {df_good_spark.columns}")
print("\nFirst 5 rows:")
display(df_good_spark.limit(5).toPandas())


StatementMeta(, 5550a4ea-07b0-45ee-ba32-9663c47d2ba5, 64, Finished, Available, Finished, False)

GOOD RECORDS (EmployeeJSONData)
Total rows: 986
Columns: ['id', 'full_name', 'f_name', 'l_name', 'dob', 'dept', 'country', 'salary', 'gender', 'job_title', 'employment_status', 'education_level', 'load_timestamp', 'source_location', 'file_type']

First 5 rows:


SynapseWidget(Synapse.DataFrame, 487f2741-3060-4e50-b41a-fc0d37c57f10)

In [34]:
print("\n" + "="*50)
print("BAD RECORDS (EmployeeJSONData_Errors)")
print("="*50)

try:
    df_bad_spark = spark.table("ODS.Employee_JSON_Data_Errors")
    if df_bad_spark.count() > 0:
        print(f"Total rows: {df_bad_spark.count()}")
        print(f"Columns: {df_bad_spark.columns}")
        print("\nFirst 5 rows:")
        display(df_bad_spark.limit(5).toPandas())
        print("\n📊 Error breakdown:")
        display(df_bad_spark.groupBy("error_type").count().toPandas())
        print("\n📊 Missing fields breakdown:")
        display(df_bad_spark.filter(col("missing_fields") != "None").groupBy("missing_fields").count().toPandas())
        print("\n📊 Extra fields breakdown:")
        display(df_bad_spark.filter(col("extra_fields") != "None").groupBy("extra_fields").count().toPandas())
    else:
        print("✅ No error records - all data was clean")
except Exception as e:
    print("✅ No error records - all data was clean")

StatementMeta(, 5550a4ea-07b0-45ee-ba32-9663c47d2ba5, 66, Finished, Available, Finished, False)


BAD RECORDS (EmployeeJSONData_Errors)
Total rows: 14
Columns: ['record_number', 'expected_count', 'actual_count', 'missing_fields', 'extra_fields', 'error_type', 'record_data', 'load_timestamp', 'source_location', 'file_type']

First 5 rows:


SynapseWidget(Synapse.DataFrame, 39392f54-4c9a-467a-9f2f-9d336b3bc28b)


📊 Error breakdown:


SynapseWidget(Synapse.DataFrame, 0f96dbc1-8f9b-4d0a-8c5a-a2317f2e56ea)


📊 Missing fields breakdown:


SynapseWidget(Synapse.DataFrame, 5eec1daa-448c-41f0-978f-3f97ea7d8f75)


📊 Extra fields breakdown:


SynapseWidget(Synapse.DataFrame, 39de489a-1c0c-466a-ab3f-4c86968f22a2)